## Sanity Check 


In [1]:
# Cell 1 — sanity check (corrected)
import os, subprocess
print(subprocess.check_output(['nvidia-smi', '-L']).decode())

CKPT_PATH   = '/kaggle/input/datasets/vathsal05/best-pt/votenet_27class_best.pt'
DATA_ROOT   = '/kaggle/input/datasets/vathsal05/synthetic-v2-60class/synthetic_v2_60class'
VOTENET_SRC = '/kaggle/input/datasets/vathsal05/votenet-source/votenet_reference'   # <- no dot-underscore

# Extra check: confirm each is the right TYPE (folder vs file)
assert os.path.isfile(CKPT_PATH),      f'CKPT should be a file: {CKPT_PATH}'
assert os.path.isdir(DATA_ROOT),       f'DATA_ROOT should be a folder: {DATA_ROOT}'
assert os.path.isdir(VOTENET_SRC),     f'VOTENET_SRC should be a folder: {VOTENET_SRC}'

# Peek inside VOTENET_SRC to confirm it has real code
contents = sorted(os.listdir(VOTENET_SRC))
print(f'\nVOTENET_SRC contains {len(contents)} items:')
for c in contents[:15]:
    print(f'  {c}')
assert 'models' in contents and 'pointnet2' in contents, 'VOTENET_SRC missing expected subdirs'
print('\nAll paths verified (with type + content check)')

GPU 0: Tesla T4 (UUID: GPU-5c61bfbf-0d77-0bf4-f432-1996fd3287f4)
GPU 1: Tesla T4 (UUID: GPU-89c03e6e-ac0b-f928-a713-3ec5f632bbad)


VOTENET_SRC contains 22 items:
  ._CODE_OF_CONDUCT.md
  ._CONTRIBUTING.md
  ._LICENSE
  ._README.md
  ._demo.py
  ._eval.py
  ._models
  ._pointnet2
  ._sunrgbd
  ._train.py
  ._utils
  CODE_OF_CONDUCT.md
  CONTRIBUTING.md
  LICENSE
  README.md

All paths verified (with type + content check)


In [2]:
# Cell 1b — install all pip deps VoteNet needs (Kaggle base image is missing several)
import subprocess
pkgs = ['plyfile', 'tensorboardX', 'trimesh',
        'opencv-python-headless', 'easydict', 'scikit-image']
for pkg in pkgs:
    r = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    print(f'{"OK  " if r.returncode == 0 else "FAIL"}: {pkg}')
print('all deps installed')

OK  : plyfile
OK  : tensorboardX
OK  : trimesh
OK  : opencv-python-headless
OK  : easydict
OK  : scikit-image
all deps installed


## Compiling PointNet2

In [3]:
# Cell 2 — copy votenet source, clean AppleDouble, patch setup.py + PyTorch API drift, compile
import os, subprocess, shutil, glob, re, sys

DEST = '/kaggle/working/votenet'

# fresh copy from input dataset to working dir
if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(VOTENET_SRC, DEST)
print(f'copied source to {DEST}')

# remove macOS AppleDouble sidecars everywhere under the tree
n_removed = 0
for root, dirs, files in os.walk(DEST):
    for f in files:
        if f.startswith('._') or f == '.DS_Store':
            os.remove(os.path.join(root, f)); n_removed += 1
    for d in list(dirs):
        if d.startswith('._'):
            shutil.rmtree(os.path.join(root, d)); n_removed += 1
            dirs.remove(d)
print(f'removed {n_removed} AppleDouble entries')

# patch setup.py — make _ext_src_root absolute (fixes -I relative path issue)
SETUP_PY = f'{DEST}/pointnet2/setup.py'
src = open(SETUP_PY).read()
old = '_ext_src_root = "_ext_src"'
new = '_ext_src_root = os.path.abspath("_ext_src")'
assert old in src, 'setup.py structure changed — check manually'
src = src.replace(old, new)
if 'import os' not in src:
    src = 'import os\n' + src
open(SETUP_PY, 'w').write(src)
print('setup.py patched (absolute _ext_src_root)')

# patch PyTorch API drift (AT_CHECK -> TORCH_CHECK, .type().is_cuda -> .is_cuda, .data<T> -> .data_ptr<T>)
EXT_SRC = f'{DEST}/pointnet2/_ext_src'
targets = (glob.glob(f'{EXT_SRC}/**/*.h', recursive=True) +
           glob.glob(f'{EXT_SRC}/**/*.cpp', recursive=True) +
           glob.glob(f'{EXT_SRC}/**/*.cu', recursive=True))
n_at, n_type, n_data = 0, 0, 0
for path in targets:
    s = open(path).read()
    orig = s
    n_at   += len(re.findall(r'\bAT_CHECK\b', s))
    s = re.sub(r'\bAT_CHECK\b', 'TORCH_CHECK', s)
    n_type += len(re.findall(r'\.type\(\)\.is_cuda\(\)', s))
    s = re.sub(r'\.type\(\)\.is_cuda\(\)', '.is_cuda()', s)
    new_s, k = re.subn(r'\.data<([^>]+)>\(\)',
                       lambda m: f'.data_ptr<{m.group(1)}>()', s)
    n_data += k
    s = new_s
    if s != orig:
        open(path, 'w').write(s)
print(f'API drift patched: AT_CHECK={n_at}, .type().is_cuda()={n_type}, .data<T>()={n_data}')

# compile pointnet2 CUDA extension
os.chdir(f'{DEST}/pointnet2')
for junk in ['build', 'dist', 'pointnet2.egg-info']:
    if os.path.exists(junk):
        shutil.rmtree(junk) if os.path.isdir(junk) else os.remove(junk)

print('\ncompiling pointnet2 (~2-3 min)...')
result = subprocess.run('python setup.py install 2>&1', shell=True,
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-1500:])
    raise RuntimeError('pointnet2 compile FAILED')
print(result.stdout[-500:])
print('compile OK')
os.chdir('/kaggle/working')

# set up import paths + smoke-test the compiled extension
for sub in ['votenet', 'votenet/models', 'votenet/utils', 'votenet/sunrgbd']:
    p = f'/kaggle/working/{sub}'
    if p not in sys.path:
        sys.path.insert(0, p)

import backbone_module, voting_module, proposal_module
print('backbone/voting/proposal imports OK')
from pointnet2 import pointnet2_utils
print('pointnet2._ext CUDA extension loaded OK')

copied source to /kaggle/working/votenet
removed 56 AppleDouble entries
setup.py patched (absolute _ext_src_root)
API drift patched: AT_CHECK=13, .type().is_cuda()=18, .data<T>()=30

compiling pointnet2 (~2-3 min)...
running egg_info
creating pointnet2.egg-info
writing pointnet2.egg-info/PKG-INFO
writing dependency_links to pointnet2.egg-info/dependency_links.txt
writing top-level names to pointnet2.egg-info/top_level.txt
writing manifest file 'pointnet2.egg-info/SOURCES.txt'
reading manifest file 'pointnet2.egg-info/SOURCES.txt'
writing manifest file 'pointnet2.egg-info/SOURCES.txt'
Copying pointnet2.egg-info to /usr/local/lib/python3.12/dist-packages/pointnet2-0.0.0-py3.12.egg-info
running install_scripts

compile OK
backbone/voting/proposal imports OK
pointnet2._ext CUDA extension loaded OK


## BAckbone Patch for Small Objects


In [4]:
# Cell 3 — patch SA layers for small-object recall
# SA1: 2048/0.2 -> 4096/0.1  (fine local geometry)
# SA2: 1024/0.4 -> 2048/0.3  (2x seed density for voting)
# SA3:  512/0.8 -> 1024/0.6  (medium radius)
# SA4:  256/1.2 ->  512/1.2  (radius unchanged, npoint doubled for context)
import importlib

BB = '/kaggle/working/votenet/models/backbone_module.py'
with open(BB) as f:
    src = f.read()

replacements = [
    ("npoint=2048,\n                radius=0.2,\n                nsample=64,",
     "npoint=4096,\n                radius=0.1,\n                nsample=64,"),
    ("npoint=1024,\n                radius=0.4,\n                nsample=32,",
     "npoint=2048,\n                radius=0.3,\n                nsample=32,"),
    ("npoint=512,\n                radius=0.8,\n                nsample=16,",
     "npoint=1024,\n                radius=0.6,\n                nsample=16,"),
    ("npoint=256,\n                radius=1.2,\n                nsample=16,",
     "npoint=512,\n                radius=1.2,\n                nsample=16,"),
]

applied = 0
for old, new in replacements:
    if old in src:
        src = src.replace(old, new)
        applied += 1
    else:
        # try loose match — indentation might be slightly different in your fork
        # find something like "npoint=<N>," near "radius=<X>,"
        print(f'WARNING: exact pattern not found, will look manually. Snippet:\n{old[:60]}...')

with open(BB, 'w') as f:
    f.write(src)

print(f'backbone_module.py patched — {applied}/4 replacements applied')

# force reload so the current kernel sees the new radii
import backbone_module
importlib.reload(backbone_module)
print('backbone_module reloaded — kernel now sees the new SA config')

# verify by grepping the patched source
with open(BB) as f:
    patched = f.read()
for radius in ['radius=0.1', 'radius=0.3', 'radius=0.6', 'radius=1.2']:
    count = patched.count(radius)
    print(f'  {radius}: {count} occurrence(s) in file')

backbone_module.py patched — 4/4 replacements applied
backbone_module reloaded — kernel now sees the new SA config
  radius=0.1: 1 occurrence(s) in file
  radius=0.3: 1 occurrence(s) in file
  radius=0.6: 1 occurrence(s) in file
  radius=1.2: 1 occurrence(s) in file


## 60 Class Configuration 

In [5]:
import numpy as np
from pathlib import Path

CLASS_NAMES = [
    'bathtub','bed','bookshelf','chair','desk','dresser','night_stand','sofa','table','toilet',
    'ammo_box','binoculars','combat_knife','flashlight','gas_mask','hand_grenade','helmet','magazine','military_radio','pistol',
    'rifle','rocket_launcher','shotgun','sniper_rifle','tactical_backpack','tactical_vest','wire_cutter',
    'axe','barbed_wire_coil','baton','canteen','claymore_mine','concrete_barrier','crossbow','duffel_bag','entrenching_shovel',
    'field_telephone','first_aid_kit','flare_gun','fuel_drum','grenade_launcher','hedgehog','jerry_can','machete','machine_gun',
    'military_boots','military_cot','military_drone','military_shield','mortar_tube','night_vision_goggles','propane_tank','rifle_case',
    'sandbag','smoke_grenade','stretcher','submachine_gun','tank_mine','tank_shell','weapon_rack',
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASS_NAMES)}
assert NUM_CLASSES == 60

DATA_ROOT = Path('/kaggle/input/datasets/vathsal05/synthetic-v2-60class/synthetic_v2_60class')
TRAIN_DIR = DATA_ROOT/'train'
VAL_DIR   = DATA_ROOT/'val'

# --- verify classes.txt ---
raw = [l.rstrip() for l in open(DATA_ROOT/'classes.txt') if l.strip()]
ref = [l.split('\t',1)[1].strip() if '\t' in l else l.strip() for l in raw]
if ref != CLASS_NAMES:
    for i,(a,b) in enumerate(zip(ref, CLASS_NAMES)):
        if a != b: print(f'  {i:2d}  dataset={a}  code={b}')
    raise AssertionError('class order mismatch')
print(f'class order verified: {NUM_CLASSES} classes')

# --- scan train bboxes for mean sizes ---
sums = np.zeros((NUM_CLASSES,3)); counts = np.zeros(NUM_CLASSES, dtype=int); n = 0
for f in sorted(TRAIN_DIR.glob('*_bbox.npy')):
    if f.name.startswith('._'): continue
    n += 1
    for box in np.load(f):
        cls = int(box[7])
        sums[cls] += [box[3], box[5], box[4]]
        counts[cls] += 1
mean_size_arr = sums / np.maximum(counts[:,None], 1)
print(f'scanned {n} train scenes | instance counts min={counts.min()} max={counts.max()} median={int(np.median(counts))}')
print(f'imbalance ratio: {counts.max()/max(counts.min(),1):.1f}x')

# --- DatasetConfig ---
class SyntheticDatasetConfig:
    def __init__(self, mean_size_arr):
        self.num_class = NUM_CLASSES
        self.num_heading_bin = 12
        self.num_size_cluster = NUM_CLASSES
        self.class2type = {i:n for i,n in enumerate(CLASS_NAMES)}
        self.type2class = {n:i for i,n in enumerate(CLASS_NAMES)}
        self.type_mean_size = {n: mean_size_arr[i] for i,n in enumerate(CLASS_NAMES)}
        self.mean_size_arr = mean_size_arr.astype(np.float32)
    def size2class(self, size, type_name):
        return self.type2class[type_name], size - self.type_mean_size[type_name]
    def class2size(self, pred_cls, residual):
        return self.mean_size_arr[pred_cls] + residual
    def angle2class(self, angle):
        angle = angle % (2*np.pi)
        per = 2*np.pi / self.num_heading_bin
        s = (angle + per/2) % (2*np.pi)
        c = int(s/per)
        return c, s - (c*per + per/2)
    def class2angle(self, pred_cls, residual, to_label_format=True):
        per = 2*np.pi / self.num_heading_bin
        a = pred_cls*per + residual
        if to_label_format and a > np.pi: a -= 2*np.pi
        return a
    def param2obb(self, center, hc, hr, sc, sr):
        obb = np.zeros(7, dtype=np.float32)
        obb[:3] = center
        obb[3:6] = self.class2size(sc, sr)
        obb[6] = self.class2angle(hc, hr)
        return obb

DC = SyntheticDatasetConfig(mean_size_arr)
print(f'\nDC ready: num_class={DC.num_class}, num_size_cluster={DC.num_size_cluster}')

# --- mean size sanity peek ---
peek = ['bed','chair','pistol','rifle','flashlight','hand_grenade','jerry_can','sandbag','concrete_barrier']
print('\nMean sizes (l x w x h, meters):')
for c in peek:
    if c in CLASS_TO_IDX:
        ms = mean_size_arr[CLASS_TO_IDX[c]]
        print(f'  {c:22s}  {ms[0]:.2f} x {ms[1]:.2f} x {ms[2]:.2f}')

class order verified: 60 classes
scanned 8000 train scenes | instance counts min=674 max=2097 median=957
imbalance ratio: 3.1x

DC ready: num_class=60, num_size_cluster=60

Mean sizes (l x w x h, meters):
  bed                     1.57 x 1.84 x 0.98
  chair                   0.32 x 0.33 x 0.55
  pistol                  0.14 x 0.14 x 0.16
  rifle                   0.21 x 0.85 x 0.27
  flashlight              0.04 x 0.05 x 0.18
  hand_grenade            0.07 x 0.12 x 0.07
  jerry_can               0.28 x 0.23 x 0.47
  sandbag                 0.64 x 0.32 x 0.12
  concrete_barrier        1.57 x 1.37 x 1.00


## SyntheticVoteNetDataset and DataLoaders

In [6]:
# Cell 5 — dataset (returns 4-channel pc) + dataloaders
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path

MAX_NUM_OBJ = 64

class SyntheticVoteNetDataset(Dataset):
    def __init__(self, split_dir, num_points=40000, augment=False, dataset_config=None):
        self.split_dir = Path(split_dir)
        self.num_points = num_points
        self.augment = augment
        self.DC = dataset_config
        self.scene_ids = sorted(
            f.name.replace('_pc.npz', '')
            for f in self.split_dir.glob('*_pc.npz')
            if not f.name.startswith('._')
        )
        print(f'{split_dir.name}: {len(self.scene_ids)} scenes')

    def __len__(self):
        return len(self.scene_ids)

    def __getitem__(self, idx):
        sid = self.scene_ids[idx]
        pc     = np.load(self.split_dir/f'{sid}_pc.npz')['pc'].astype(np.float32)
        bboxes = np.load(self.split_dir/f'{sid}_bbox.npy').astype(np.float32)
        votes  = np.load(self.split_dir/f'{sid}_votes.npz')['votes'].astype(np.float32)

        N = pc.shape[0]
        if N != self.num_points:
            choice = np.random.choice(N, self.num_points, replace=(N < self.num_points))
            pc = pc[choice]; votes = votes[choice]

        if self.augment:
            theta = np.random.uniform(-np.pi/6, np.pi/6)
            c, s = np.cos(theta), np.sin(theta)
            R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], dtype=np.float32)
            pc[:, :3]     = pc[:, :3]     @ R.T
            bboxes[:, :3] = bboxes[:, :3] @ R.T
            bboxes[:, 6] += theta
            for k in range(3):
                votes[:, 1+k*3:4+k*3] = votes[:, 1+k*3:4+k*3] @ R.T

            scale = np.random.uniform(0.9, 1.1)
            pc[:, :3]     *= scale
            bboxes[:, :6] *= scale
            for k in range(3):
                votes[:, 1+k*3:4+k*3] *= scale

            if np.random.rand() < 0.5:
                pc[:, 0]     = -pc[:, 0]
                bboxes[:, 0] = -bboxes[:, 0]
                bboxes[:, 6] = -bboxes[:, 6]
                for k in range(3):
                    votes[:, 1+k*3] = -votes[:, 1+k*3]

        center_label            = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        heading_class_label     = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        heading_residual_label  = np.zeros(MAX_NUM_OBJ, dtype=np.float32)
        size_class_label        = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        size_residual_label     = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        sem_cls_label           = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        box_label_mask          = np.zeros(MAX_NUM_OBJ, dtype=np.float32)

        for i in range(min(len(bboxes), MAX_NUM_OBJ)):
            box = bboxes[i]
            cls = int(box[7])
            center_label[i] = box[:3]
            hc, hr = self.DC.angle2class(float(box[6]))
            heading_class_label[i]    = hc
            heading_residual_label[i] = hr
            size_class_label[i]       = cls
            reordered = np.array([box[3], box[5], box[4]], dtype=np.float32)
            size_residual_label[i]    = reordered - self.DC.mean_size_arr[cls]
            sem_cls_label[i]          = cls
            box_label_mask[i]         = 1

        return {
            'point_clouds':           pc[:, :4].copy(),   # <-- 4 channels only (XYZ + 1 feat)
            'center_label':           center_label,
            'heading_class_label':    heading_class_label,
            'heading_residual_label': heading_residual_label,
            'size_class_label':       size_class_label,
            'size_residual_label':    size_residual_label,
            'sem_cls_label':          sem_cls_label,
            'box_label_mask':         box_label_mask,
            'vote_label':             votes[:, 1:10],
            'vote_label_mask':        votes[:, 0].astype(np.int64),
            'scan_idx':               np.array(idx, dtype=np.int64),
        }

train_ds = SyntheticVoteNetDataset(TRAIN_DIR, num_points=40000, augment=True,  dataset_config=DC)
val_ds   = SyntheticVoteNetDataset(VAL_DIR,   num_points=40000, augment=False, dataset_config=DC)

sample = train_ds[0]
print(f'\nsample point_clouds shape: {sample["point_clouds"].shape}  (should be (40000, 4))')
print(f'num valid boxes in sample 0: {int(sample["box_label_mask"].sum())}')

train: 8000 scenes
val: 1500 scenes

sample point_clouds shape: (40000, 4)  (should be (40000, 4))
num valid boxes in sample 0: 9


## size-weighted classification loss

In [7]:
# Cell 6 — size-weighted semantic classification loss patch
import numpy as np
import torch
import builtins
import importlib

CLASS_REAL_SIZE = {
    'bed':2.0,'table':1.5,'sofa':2.0,'chair':0.55,'toilet':0.6,'desk':1.4,
    'dresser':1.0,'night_stand':0.55,'bookshelf':0.85,'bathtub':1.6,
    'ammo_box':0.35,'binoculars':0.22,'combat_knife':0.30,'flashlight':0.18,
    'gas_mask':0.28,'hand_grenade':0.12,'helmet':0.28,'magazine':0.18,
    'military_radio':0.30,'pistol':0.22,'rifle':0.95,'rocket_launcher':1.20,
    'shotgun':0.95,'sniper_rifle':1.20,'tactical_backpack':0.55,
    'tactical_vest':0.50,'wire_cutter':0.25,'axe':0.60,'barbed_wire_coil':0.90,
    'baton':0.55,'canteen':0.20,'claymore_mine':0.22,'concrete_barrier':2.00,
    'crossbow':0.75,'duffel_bag':0.80,'entrenching_shovel':0.60,
    'field_telephone':0.30,'first_aid_kit':0.30,'flare_gun':0.25,
    'fuel_drum':0.90,'grenade_launcher':0.75,'hedgehog':1.40,'jerry_can':0.47,
    'machete':0.65,'machine_gun':1.25,'military_boots':0.32,'military_cot':1.90,
    'military_drone':0.90,'military_shield':1.30,'mortar_tube':1.30,
    'night_vision_goggles':0.20,'propane_tank':0.60,'rifle_case':1.20,
    'sandbag':0.65,'smoke_grenade':0.15,'stretcher':2.10,'submachine_gun':0.60,
    'tank_mine':0.33,'tank_shell':0.90,'weapon_rack':1.80,
}

# Sanity: every class in CLASS_NAMES has a size entry
missing = [c for c in CLASS_NAMES if c not in CLASS_REAL_SIZE]
assert not missing, f'missing sizes for: {missing}'
sizes = np.array([CLASS_REAL_SIZE[c] for c in CLASS_NAMES])

# Inverse-sqrt-size weighting: small objects get bigger CE weight
# clamp to [0.5, 3.0] so no single class dominates the loss
w = np.clip(np.sqrt(np.median(sizes) / sizes), 0.5, 3.0)
SEM_W = torch.tensor(w, dtype=torch.float32).cuda()

print(f'median class size: {np.median(sizes):.2f} m')
print(f'weight range: [{w.min():.2f}, {w.max():.2f}]\n')

print('Classes with weight > 1.5 (UP-weighted — small objects):')
for c, x, sz in sorted(zip(CLASS_NAMES, w, sizes), key=lambda t: -t[1]):
    if x > 1.5:
        print(f'  {c:24s} w={x:.2f}  size={sz:.2f}m')

print('\nClasses with weight < 0.7 (DOWN-weighted — large objects):')
for c, x, sz in sorted(zip(CLASS_NAMES, w, sizes), key=lambda t: t[1]):
    if x < 0.7:
        print(f'  {c:24s} w={x:.2f}  size={sz:.2f}m')

# ---- Patch loss_helper.py to use the weights ----
LH = '/kaggle/working/votenet/models/loss_helper.py'
with open(LH) as f:
    src = f.read()

old = "criterion_sem_cls = nn.CrossEntropyLoss(reduction='none')"
new = ("import builtins\n    "
       "criterion_sem_cls = nn.CrossEntropyLoss("
       "weight=getattr(builtins, 'SEM_CLS_WEIGHTS', None), reduction='none')")

if old not in src:
    print('\nWARNING: exact pattern not found. Candidates in file:')
    import re
    for m in re.finditer(r'criterion_sem_cls\s*=\s*nn\.CrossEntropyLoss[^)]*\)', src):
        print(f'  {m.group(0)}')
    raise RuntimeError('adjust the pattern to match your fork and re-run')

src = src.replace(old, new)
with open(LH, 'w') as f:
    f.write(src)
print('\nloss_helper.py patched with weighted CrossEntropyLoss')

# ---- Set global weights (kernel-scoped) ----
builtins.SEM_CLS_WEIGHTS = SEM_W
print(f'builtins.SEM_CLS_WEIGHTS set — shape={tuple(SEM_W.shape)}, device={SEM_W.device}')

# Reload loss_helper if it was already imported anywhere
try:
    import loss_helper
    importlib.reload(loss_helper)
    print('loss_helper reloaded — weighted loss active in kernel')
except ImportError:
    print('loss_helper not previously imported (will use patched version on first import)')

median class size: 0.60 m
weight range: [0.53, 2.24]

Classes with weight > 1.5 (UP-weighted — small objects):
  hand_grenade             w=2.24  size=0.12m
  smoke_grenade            w=2.00  size=0.15m
  flashlight               w=1.83  size=0.18m
  magazine                 w=1.83  size=0.18m
  canteen                  w=1.73  size=0.20m
  night_vision_goggles     w=1.73  size=0.20m
  binoculars               w=1.65  size=0.22m
  pistol                   w=1.65  size=0.22m
  claymore_mine            w=1.65  size=0.22m
  wire_cutter              w=1.55  size=0.25m
  flare_gun                w=1.55  size=0.25m

Classes with weight < 0.7 (DOWN-weighted — large objects):
  stretcher                w=0.53  size=2.10m
  bed                      w=0.55  size=2.00m
  sofa                     w=0.55  size=2.00m
  concrete_barrier         w=0.55  size=2.00m
  military_cot             w=0.56  size=1.90m
  weapon_rack              w=0.58  size=1.80m
  bathtub                  w=0.61  size=1.60m
 

In [8]:
# Cell 6c — verify install (without the buggy version check) + clear stale imports
import plyfile
import tensorboardX
print('plyfile: OK (imports without error)')
print('tensorboardX: OK')

# Clear the cached broken imports so Cell 7's next attempt is fresh
import sys
cleared = []
for mod in list(sys.modules):
    if mod in ('pc_util', 'dump_helper', 'loss_helper', 'votenet',
               'models.votenet', 'backbone_module', 'voting_module',
               'proposal_module'):
        del sys.modules[mod]
        cleared.append(mod)
print(f'cleared cached imports: {cleared}')

plyfile: OK (imports without error)
tensorboardX: OK
cleared cached imports: ['voting_module', 'proposal_module', 'backbone_module', 'loss_helper']


In [9]:
# Cell 6d — install all remaining VoteNet-adjacent deps we might need
import subprocess

pkgs = ['trimesh', 'opencv-python-headless', 'easydict', 'scikit-image']
for pkg in pkgs:
    r = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True, text=True)
    print(f'{"installed" if r.returncode == 0 else "FAILED":10s}: {pkg}')
    if r.returncode != 0:
        print(f'  stderr: {r.stderr[-300:]}')

# Verify by importing (each in its own try so one failure doesn't stop the rest)
for name in ['trimesh', 'cv2', 'easydict', 'skimage']:
    try:
        __import__(name)
        print(f'  import OK: {name}')
    except ImportError as e:
        print(f'  import FAIL: {name} — {e}')

# Clear cached broken imports again
import sys
for mod in list(sys.modules):
    if mod in ('pc_util', 'dump_helper', 'loss_helper', 'votenet',
               'models.votenet', 'backbone_module', 'voting_module',
               'proposal_module'):
        del sys.modules[mod]
print('\ncached imports cleared')

installed : trimesh
installed : opencv-python-headless
installed : easydict
installed : scikit-image
  import OK: trimesh
  import OK: cv2
  import OK: easydict
  import OK: skimage

cached imports cleared


## model instantiation and selective weight transfer

In [10]:
# Cell 7 — build VoteNet + selectively transfer weights + smoke test
import torch, os

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

try:
    from votenet import VoteNet
except ImportError:
    from models.votenet import VoteNet

model = VoteNet(
    num_class=DC.num_class,
    num_heading_bin=DC.num_heading_bin,
    num_size_cluster=DC.num_size_cluster,
    mean_size_arr=DC.mean_size_arr,
    num_proposal=512,
    input_feature_dim=1,
    vote_factor=1,
    sampling='vote_fps'
).to(device)
print(f'VoteNet built — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

CKPT_PATH = '/kaggle/input/datasets/vathsal05/best-pt/votenet_27class_best.pt'
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt

model_state = model.state_dict()
transferable = {k: v for k, v in state.items()
                if k in model_state and v.shape == model_state[k].shape}
result = model.load_state_dict(transferable, strict=False)
print(f'transferred {len(transferable)}/{len(model_state)} layers  '
      f'({100*len(transferable)/len(model_state):.1f}%)')
print(f'missing (re-init from scratch): {len(result.missing_keys)}')

# smoke test with real 4-channel data
model.eval()
sample = train_ds[0]
real_input = {'point_clouds': torch.from_numpy(sample['point_clouds']).unsqueeze(0).to(device)}
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    ep = model(real_input)
mem_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'forward OK — {len(ep)} output keys | peak GPU mem (bs=1): {mem_gb:.2f} GB')

torch.save({
    'epoch': 0, 'model_state_dict': model.state_dict(),
    'best_val_loss': float('inf'),
}, '/kaggle/working/votenet_60class_baseline.pt')
print('baseline saved')

device: cuda:0
VoteNet built — 1.0M params
transferred 144/146 layers  (98.6%)
missing (re-init from scratch): 2
forward OK — 29 output keys | peak GPU mem (bs=1): 0.36 GB
baseline saved


## Tranining Loop

In [ ]:
# Cell 8 — training loop with resume-safe checkpointing
import torch, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from pathlib import Path

# ---- rebuild loaders at bs=8 ----
BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=False)
print(f'train: {len(train_loader)} batches | val: {len(val_loader)} batches | bs={BATCH_SIZE}')

from loss_helper import get_loss

LABEL_KEYS = ['center_label','heading_class_label','heading_residual_label',
              'size_class_label','size_residual_label','sem_cls_label',
              'box_label_mask','vote_label','vote_label_mask']

EPOCHS, LR_INIT, LR_MIN = 30, 1e-4, 1e-6
optimizer = AdamW(model.parameters(), lr=LR_INIT, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

# ---- resume if last checkpoint exists ----
start_ep, best_val_loss, history = 0, float('inf'), []
RESUME = Path('/kaggle/working/votenet_60class_last.pt')
if RESUME.exists():
    ck = torch.load(RESUME, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    scheduler.load_state_dict(ck['scheduler_state_dict'])
    start_ep = ck['epoch']
    best_val_loss = ck.get('best_val_loss', float('inf'))
    history = ck.get('history', [])
    print(f'resumed from ep {start_ep} (best_val_loss={best_val_loss:.4f})')
else:
    print('starting fresh from Phase 8 baseline')

def run_epoch(loader, training):
    model.train() if training else model.eval()
    total, n = 0.0, 0
    for batch in loader:
        bg = {k: v.to(device, non_blocking=True) for k, v in batch.items()
              if isinstance(v, torch.Tensor)}
        if training:
            optimizer.zero_grad()
            out = model(bg)
            for k in LABEL_KEYS: out[k] = bg[k]
            loss, out = get_loss(out, DC)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                out = model(bg)
                for k in LABEL_KEYS: out[k] = bg[k]
                loss, out = get_loss(out, DC)
        total += loss.item(); n += 1
    return total / max(n, 1)

for ep in range(start_ep, EPOCHS):
    t0 = time.time()
    tr = run_epoch(train_loader, training=True)
    vl = run_epoch(val_loader, training=False)
    scheduler.step()
    dt = (time.time() - t0) / 60

    is_best = vl < best_val_loss
    if is_best: best_val_loss = vl
    lr = scheduler.get_last_lr()[0]

    history.append({'epoch': ep+1, 'train': tr, 'val': vl})
    print(f'ep {ep+1:2d}/{EPOCHS}  train={tr:.4f}  val={vl:.4f}  lr={lr:.2e}  '
          f'time={dt:.1f}min  {"*BEST*" if is_best else ""}')

    torch.save({
        'epoch': ep+1, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'history': history,
    }, '/kaggle/working/votenet_60class_last.pt')

    if is_best:
        torch.save({
            'epoch': ep+1, 'model_state_dict': model.state_dict(),
            'val_loss': vl, 'history': history,
        }, '/kaggle/working/votenet_60class_best.pt')

print(f'\nDone. Best val_loss = {best_val_loss:.4f}')

## full evaluation: mAP + per-class + small-object F1

In [11]:
# Cell 9 — evaluate best checkpoint on val set (mAP + per-class + small-obj F1)
import torch, pickle, numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

# ---- load best trained checkpoint ----
BEST_CKPT = '/kaggle/input/datasets/vathsal05/60-best/votenet_60class_best.pt'
if not Path(BEST_CKPT).exists():
    raise FileNotFoundError(f'{BEST_CKPT} missing — check the notebook Output tab '
                            'of your committed version and re-download if needed')
ck = torch.load(BEST_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ck['model_state_dict'])
model.eval()
print(f'loaded best (val_loss={ck.get("val_loss","?"):.4f})')

# ---- eval loader ----
EVAL_BS = 4
val_loader_eval = DataLoader(val_ds, batch_size=EVAL_BS, shuffle=False,
                             num_workers=2, pin_memory=True, drop_last=False)
print(f'val batches: {len(val_loader_eval)} @ bs={EVAL_BS}')

# ---- mAP tools ----
from ap_helper import APCalculator, parse_predictions, parse_groundtruths

CONFIG_DICT = {'remove_empty_box': False, 'use_3d_nms': True, 'nms_iou': 0.25,
               'use_old_type_nms': False, 'cls_nms': True,
               'per_class_proposal': True, 'conf_thresh': 0.05,
               'dataset_config': DC}

LABEL_KEYS = ['center_label','heading_class_label','heading_residual_label',
              'size_class_label','size_residual_label','sem_cls_label',
              'box_label_mask','vote_label','vote_label_mask']

ap25 = APCalculator(0.25, DC.class2type)
ap50 = APCalculator(0.50, DC.class2type)

all_preds, all_gts = [], []
print('\nrunning inference...')
for i, batch in enumerate(val_loader_eval):
    bg = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
    with torch.no_grad():
        out = model(bg)
    for k in LABEL_KEYS: out[k] = bg[k]
    preds = parse_predictions(out, CONFIG_DICT)
    gts = parse_groundtruths(out, CONFIG_DICT)
    ap25.step(preds, gts); ap50.step(preds, gts)
    all_preds += preds; all_gts += gts
    if (i+1) % 25 == 0 or i == len(val_loader_eval)-1:
        print(f'  {i+1}/{len(val_loader_eval)}')

# ---- headline mAP ----
print('\n' + '='*50)
for name, calc in [('mAP@0.25', ap25), ('mAP@0.50', ap50)]:
    m = calc.compute_metrics()
    print(f'\n{name}: {m["mAP"]:.4f}')
    for c in CLASS_NAMES:
        key = f'{c} Average Precision'
        if key in m:
            sz = CLASS_REAL_SIZE.get(c, 0)
            tag = ' *small*' if sz < 0.35 else ''
            print(f'  {c:24s} {m[key]:.3f}   ({sz:.2f}m){tag}')

# ---- small-object F1 @ 10cm center distance ----
print('\n' + '='*50)
print('Small-object F1 @ 10cm center distance:')
SMALL = [c for c in CLASS_NAMES if CLASS_REAL_SIZE.get(c, 999) < 0.35]

def centers(entries, cls_idx, min_score=None):
    out = []
    for e in entries:
        if e[0] != cls_idx: continue
        if min_score is not None and len(e) > 2 and e[2] < min_score: continue
        out.append(np.asarray(e[1]).reshape(-1,3).mean(axis=0))
    return out

print(f'{"class":22s} {"P":>7s} {"R":>7s} {"F1":>7s}')
for c in SMALL:
    ci = CLASS_TO_IDX[c]
    tp = fp = fn = 0
    for preds, gts in zip(all_preds, all_gts):
        pc = centers(preds, ci, min_score=0.5)
        gc = centers(gts, ci)
        used = set()
        for p in pc:
            d = [np.linalg.norm(p - g) for g in gc]
            j = int(np.argmin(d)) if d else -1
            if j >= 0 and d[j] < 0.10 and j not in used:
                tp += 1; used.add(j)
            else:
                fp += 1
        fn += len(gc) - len(used)
    P = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1)
    F1 = 2*P*R / max(P+R, 1e-9)
    print(f'  {c:20s}  {P:.3f}  {R:.3f}  {F1:.3f}')

# ---- save predictions for Plotly viz ----
pickle.dump({'preds': all_preds, 'gts': all_gts,
             'val_loss': float(ck.get('val_loss', -1)),
             'class_names': CLASS_NAMES,
             'best_ckpt_epoch': ck.get('epoch')},
            open('/kaggle/working/val_predictions_60class.pkl', 'wb'))
print('\nsaved: /kaggle/working/val_predictions_60class.pkl')

loaded best (val_loss=6.9306)
val batches: 375 @ bs=4

running inference...
  25/375
  50/375
  75/375
  100/375
  125/375
  150/375
  175/375
  200/375
  225/375
  250/375
  275/375
  300/375
  325/375
  350/375
  375/375

0 0.68971233449219
1 0.9309648077146788
2 0.5666558589164677
3 0.9727882192872483
4 0.76495922258354
5 0.9529846757906728
6 0.9748876153277853
7 0.5198396783920869
8 0.8555957556891128
9 0.9358954731913827
10 0.530794774492759
11 0.338788780976938
12 0.0005523039233746689
13 0.08002076373973421
14 0.7454785305599697
15 0.04501624513132367
16 0.8876667498336663
17 0.006853435727274633
18 0.25716231244799964
19 0.016542598695888994
20 0.025758950601904657
21 0.16204827130414556
22 0.01680315499092895
23 0.03160232273314105
24 0.9302059545823057
25 0.8742552388321578
26 0.0007479825658648641
27 0.060816726108742156
28 0.9976149151561798
29 0.0012985090142053097
30 0.366221907534489
31 0.27258012626178896
32 0.8367697701149389
33 0.44833882420780574
34 0.963323267003696